# 🧪 Séance 1 — Prédire la toxicité d'une molécule avec un Random Forest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/IA_Biotech/blob/main/notebooks/S1_toxicite_random_forest.ipynb)

**Module : Intelligence Artificielle et Biotechnologie — Atelier 1 / 5 (2h)**

## 🧬 Le problème biologique

Avant de tester une molécule sur des cellules ou des animaux, l'industrie pharmaceutique et les agences
sanitaires veulent **éliminer au plus tôt les composés toxiques**. Le programme américain **Tox21**
(NIH, EPA, FDA) a testé ~8 000 molécules (médicaments, pesticides, additifs…) sur 12 tests cellulaires.

Nous allons nous intéresser au test **NR-AhR** : la molécule active-t-elle le **récepteur aryl
hydrocarbure (AhR)** ? Ce récepteur est activé par des polluants comme les **dioxines** ou les
**hydrocarbures aromatiques polycycliques** (fumée, goudron) et son activation est associée à des
effets toxiques (cancérogénèse, perturbation immunitaire).

> **Question :** peut-on prédire, *uniquement à partir de propriétés physico-chimiques simples*
> (masse, lipophilie, nombre de cycles aromatiques…), si une molécule active AhR ?

## 🎯 Objectifs de l'atelier
1. Charger et **nettoyer** un fichier CSV réel avec Pandas
2. Calculer des **descripteurs physico-chimiques** à partir de la structure chimique (SMILES)
3. **Visualiser** les distributions et repérer les variables discriminantes
4. Entraîner un **Random Forest** et l'évaluer correctement (matrice de confusion, ROC-AUC)
5. Identifier les **variables les plus déterminantes** (feature importance) et les interpréter

## 🧭 Légende
| Icône | Signification |
|---|---|
| ▶️ | Code fourni : **exécutez** la cellule (`Maj + Entrée`) et lisez le résultat |
| ✍️ | **À vous** : modifiez un paramètre ou complétez quelques lignes |
| 🔎 | **Interprétez** : répondez à la question dans la cellule de texte |
| 💡 | Solution (cellule masquée : double-cliquez pour voir le code) |

⏱️ **Déroulé indicatif :** installation (5 min) · exploration & nettoyage (30 min) · descripteurs & visualisation (20 min) · modèle & évaluation (40 min) · importance des variables & défis (25 min)

---
## 0. Installation et imports ▶️

Google Colab contient déjà Pandas, Scikit-learn et Matplotlib. Nous ajoutons seulement **RDKit**,
la bibliothèque de référence en chimie-informatique (environ 30 secondes).

In [ ]:
# ▶️ Installation de RDKit (chimie-informatique)
%pip install -q rdkit

In [ ]:
# ▶️ Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, Draw

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance

RDLogger.DisableLog("rdApp.*")          # masque les avertissements de RDKit
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
print("✅ Environnement prêt")

---
## 1. Charger les données ▶️

Le fichier `tox21.csv.gz` contient une ligne par molécule :
- `smiles` : la structure chimique écrite en texte (format **SMILES**, ex. `CCO` = éthanol)
- 12 colonnes de résultats de tests : `1` = actif (toxique), `0` = inactif, **vide = non testé**

In [ ]:
# ▶️ Téléchargement du jeu de données Tox21 (source : MoleculeNet / DeepChem)
URLS_TOX21 = [
    "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz",
    "https://raw.githubusercontent.com/nevermind78/IA_Biotech/main/data/tox21.csv.gz",  # copie de secours
]

df_brut = None
for url in URLS_TOX21:
    try:
        df_brut = pd.read_csv(url)
        print("✅ Données chargées depuis :", url)
        break
    except Exception as e:
        print("⚠️ Échec pour", url, "→", type(e).__name__)

print("Dimensions (lignes, colonnes) :", df_brut.shape)
df_brut.head()

In [ ]:
# ▶️ Informations générales : types des colonnes et valeurs non nulles
df_brut.info()

### 🔎 Question 1
Combien de molécules contient le fichier ? Pourquoi certaines colonnes de tests ont-elles **moins de
valeurs non nulles** que d'autres ?

*Votre réponse :* …

In [ ]:
# ▶️ Valeurs manquantes par test (= molécules non testées)
manquants = df_brut.isna().sum().sort_values()
manquants.plot(kind="barh", figsize=(7, 4), color="tab:orange")
plt.title("Nombre de valeurs manquantes par colonne")
plt.xlabel("Nombre de molécules non testées")
plt.show()

---
## 2. Nettoyer les données

Un jeu de données réel n'est jamais « propre ». Nous allons :
1. garder **une seule cible** (le test choisi) et la colonne `smiles` ;
2. supprimer les molécules **non testées** pour cette cible ;
3. retirer les **sels et contre-ions** (ex. `[Na+]`) pour ne garder que la molécule principale ;
4. supprimer les **doublons**.

In [ ]:
# ✍️ Choisissez le test à prédire (gardez NR-AhR la première fois)
CIBLE = "NR-AhR"  #@param ["NR-AhR", "SR-MMP", "SR-p53", "NR-AR", "NR-ER", "SR-ARE", "SR-HSE", "NR-Aromatase"]

df = df_brut[["smiles", CIBLE]].copy()
df = df.rename(columns={CIBLE: "toxique"})
print("Avant nettoyage :", len(df), "molécules")

# Étape 2 : supprimer les molécules non testées
df = df.dropna(subset=["toxique"])
df["toxique"] = df["toxique"].astype(int)
print("Après suppression des non testées :", len(df))

In [ ]:
# ▶️ Étape 3 : retirer les sels (on garde le plus gros fragment de la formule SMILES)
def plus_gros_fragment(smiles):
    """'CC(=O)[O-].[Na+]'  →  'CC(=O)[O-]'  (les fragments sont séparés par un point)."""
    fragments = smiles.split(".")
    return max(fragments, key=len)

exemple = df.loc[df["smiles"].str.contains(r"\.", regex=True), "smiles"].iloc[0]
print("Exemple avec sel :", exemple, "  →  ", plus_gros_fragment(exemple))

df["smiles"] = df["smiles"].apply(plus_gros_fragment)

### ✍️ Exercice 1 — Supprimer les doublons
Après le retrait des sels, certaines molécules apparaissent **plusieurs fois**.
1. Comptez les doublons de la colonne `smiles` avec `df.duplicated(...)`.
2. Supprimez-les avec `df.drop_duplicates(...)` en gardant la première occurrence.

*Indice :* `df.duplicated(subset="smiles").sum()`

In [ ]:
# ✍️ À vous ! Remplacez None par votre code
nb_doublons = None   # ← comptez les doublons ici

if nb_doublons is None:
    print("⚠️ Complétez la ligne 'nb_doublons = ...' puis ré-exécutez la cellule")
else:
    print("Nombre de doublons :", nb_doublons)

In [ ]:
#@title 💡 Solution de l'exercice 1 (exécutez-la dans tous les cas)
nb_doublons = df.duplicated(subset="smiles").sum()
print("Nombre de doublons :", nb_doublons)
df = df.drop_duplicates(subset="smiles", keep="first").reset_index(drop=True)
print("Molécules uniques :", len(df))

In [ ]:
# ▶️ Répartition des classes
print(df["toxique"].value_counts())
print(f"\nProportion de molécules toxiques : {df['toxique'].mean():.1%}")
df["toxique"].value_counts().plot(kind="bar", color=["tab:blue", "tab:red"], figsize=(4, 3))
plt.xticks([0, 1], ["Inactif (0)", "Actif (1)"], rotation=0)
plt.title(f"Répartition des classes — {CIBLE}")
plt.show()

### 🔎 Question 2
Les classes sont-elles **équilibrées** ? Un modèle qui répondrait toujours « non toxique » aurait
quelle exactitude (accuracy) ? Gardez ce chiffre en tête pour la partie 4 !

*Votre réponse :* …

---
## 3. Des molécules aux nombres : les descripteurs ▶️

Un algorithme d'apprentissage ne lit pas une formule chimique : il lui faut des **nombres**.
Nous calculons pour chaque molécule des **descripteurs physico-chimiques** classiques en chimie
médicinale (cf. règle de Lipinski) :

| Descripteur | Signification biologique |
|---|---|
| `masse_molaire` | taille de la molécule (g/mol) |
| `logP` | lipophilie : tendance à traverser les membranes / s'accumuler dans les graisses |
| `TPSA` | surface polaire : liée à l'absorption et au passage des barrières |
| `donneurs_H`, `accepteurs_H` | capacité à former des liaisons hydrogène |
| `liaisons_rotatives` | flexibilité de la molécule |
| `cycles_aromatiques`, `nb_cycles` | molécules planes et aromatiques (comme les dioxines) |
| `fraction_sp3` | proportion de carbones « 3D » (non plans) |
| `halogenes` | nombre d'atomes F, Cl, Br, I (fréquents dans les polluants) |

In [ ]:
# ▶️ Fonction fournie : SMILES → dictionnaire de descripteurs
def calculer_descripteurs(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:                      # SMILES invalide
        return None
    return {
        "masse_molaire": Descriptors.MolWt(mol),
        "logP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "donneurs_H": Lipinski.NumHDonors(mol),
        "accepteurs_H": Lipinski.NumHAcceptors(mol),
        "liaisons_rotatives": Lipinski.NumRotatableBonds(mol),
        "cycles_aromatiques": rdMolDescriptors.CalcNumAromaticRings(mol),
        "nb_cycles": rdMolDescriptors.CalcNumRings(mol),
        "fraction_sp3": rdMolDescriptors.CalcFractionCSP3(mol),
        "atomes_lourds": mol.GetNumHeavyAtoms(),
        "halogenes": sum(a.GetSymbol() in ("F", "Cl", "Br", "I") for a in mol.GetAtoms()),
        "charge_formelle": Chem.GetFormalCharge(mol),
    }

# Test sur la caféine
print(calculer_descripteurs("Cn1cnc2c1c(=O)n(C)c(=O)n2C"))
Draw.MolToImage(Chem.MolFromSmiles("Cn1cnc2c1c(=O)n(C)c(=O)n2C"), size=(250, 180))

In [ ]:
# ▶️ Calcul pour toutes les molécules (≈ 20 secondes)
descripteurs = df["smiles"].apply(calculer_descripteurs)
print("SMILES invalides :", descripteurs.isna().sum())

df = df[descripteurs.notna()].reset_index(drop=True)
X_desc = pd.DataFrame(descripteurs.dropna().tolist())
data = pd.concat([df, X_desc], axis=1)
COLONNES_X = list(X_desc.columns)

print("Tableau final :", data.shape)
data.head()

In [ ]:
# ▶️ Statistiques descriptives
data[COLONNES_X].describe().T.round(2)

### Visualiser les distributions par classe ▶️

In [ ]:
# ▶️ Histogrammes de chaque descripteur, toxiques (rouge) vs non toxiques (bleu)
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.ravel(), COLONNES_X):
    sns.histplot(data=data, x=col, hue="toxique", stat="density", common_norm=False,
                 bins=30, ax=ax, palette={0: "tab:blue", 1: "tab:red"}, element="step")
    ax.set_title(col)
plt.suptitle(f"Distribution des descripteurs selon la classe ({CIBLE})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ▶️ Moyenne de chaque descripteur par classe
data.groupby("toxique")[COLONNES_X].mean().T.round(2)

### 🔎 Question 3
D'après les histogrammes et le tableau des moyennes, **quels descripteurs semblent différer le plus**
entre molécules actives et inactives ? Est-ce cohérent avec le fait qu'AhR soit activé par des
molécules **planes, aromatiques et chlorées** comme les dioxines ?

*Votre réponse :* …

In [ ]:
# ▶️ Corrélations entre descripteurs (variables redondantes ?)
plt.figure(figsize=(9, 7))
sns.heatmap(data[COLONNES_X].corr(), annot=True, fmt=".1f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation des descripteurs")
plt.show()

---
## 4. Entraîner et évaluer un modèle

### 4.1 Séparer apprentissage et test ▶️
On garde **20 % des molécules de côté** : le modèle ne les verra jamais pendant l'entraînement.
`stratify=y` conserve la même proportion de toxiques dans les deux ensembles.

In [ ]:
X = data[COLONNES_X]
y = data["toxique"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Apprentissage :", X_train.shape, f"— toxiques : {y_train.mean():.1%}")
print("Test          :", X_test.shape, f"— toxiques : {y_test.mean():.1%}")

### 4.2 Le piège de l'exactitude : un modèle « idiot » ▶️

In [ ]:
idiot = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
y_pred_idiot = idiot.predict(X_test)
print(f"Exactitude (accuracy) du modèle idiot : {accuracy_score(y_test, y_pred_idiot):.1%}")
print(f"Rappel (toxiques détectés)            : {recall_score(y_test, y_pred_idiot):.1%}")

😮 Un modèle qui répond **toujours « non toxique »** atteint une exactitude élevée… mais ne détecte
**aucune** molécule toxique ! Sur des données déséquilibrées, il faut regarder le **rappel**, la
**précision**, le **F1** et l'**AUC ROC**.

### 4.3 Random Forest ▶️ ✍️
Une forêt aléatoire = **beaucoup d'arbres de décision** entraînés sur des sous-échantillons différents,
qui **votent**. `class_weight="balanced"` donne plus de poids aux molécules toxiques (classe rare).

In [ ]:
# ✍️ Hyperparamètres : commencez avec les valeurs par défaut, vous les modifierez au défi 2
N_ARBRES = 200        #@param {type:"slider", min:10, max:500, step:10}
PROFONDEUR_MAX = 12   #@param {type:"slider", min:2, max:30, step:1}

rf = RandomForestClassifier(
    n_estimators=N_ARBRES,
    max_depth=PROFONDEUR_MAX,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print("✅ Modèle entraîné")

In [ ]:
# ▶️ Fonction d'évaluation réutilisable
def evaluer(modele, X_, y_, seuil=0.5, nom="Modèle"):
    proba = modele.predict_proba(X_)[:, 1]
    y_pred = (proba >= seuil).astype(int)
    scores = {
        "exactitude": accuracy_score(y_, y_pred),
        "exactitude_équilibrée": balanced_accuracy_score(y_, y_pred),
        "précision": precision_score(y_, y_pred, zero_division=0),
        "rappel": recall_score(y_, y_pred),
        "F1": f1_score(y_, y_pred),
        "AUC_ROC": roc_auc_score(y_, proba),
    }
    return pd.Series(scores, name=nom).round(3)

pd.concat([evaluer(rf, X_train, y_train, nom="Apprentissage"),
           evaluer(rf, X_test, y_test, nom="Test")], axis=1)

In [ ]:
# ▶️ Matrice de confusion et courbe ROC sur le jeu de test
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, display_labels=["Inactif", "Toxique"],
                                      cmap="Blues", ax=axes[0])
axes[0].set_title("Matrice de confusion (seuil 0,5)")
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=axes[1], name="Random Forest")
axes[1].plot([0, 1], [0, 1], "k--", label="Hasard (AUC = 0,5)")
axes[1].legend()
axes[1].set_title("Courbe ROC")
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf.predict(X_test), target_names=["Inactif", "Toxique"]))

### 🔎 Question 4
1. Lisez la matrice de confusion : combien de molécules toxiques ont été **manquées** (faux négatifs) ?
   Combien de molécules saines ont été **faussement alarmées** (faux positifs) ?
2. En **criblage toxicologique**, quelle erreur est la plus grave ? Pourquoi ?
3. Comparez l'AUC sur l'apprentissage et sur le test. Le modèle **sur-apprend**-il ?

*Votre réponse :* …

### ✍️ Exercice 2 — Ajuster le seuil de décision
Par défaut, une molécule est déclarée toxique si sa probabilité ≥ 0,5. En toxicologie, on préfère
**ne rater aucun toxique**, quitte à avoir plus de fausses alertes.

Modifiez `SEUIL` (essayez 0.5, 0.3, 0.2, 0.1) et observez l'évolution du **rappel** et de la **précision**.

In [ ]:
# ✍️ À vous : changez la valeur du seuil
SEUIL = 0.3  #@param {type:"slider", min:0.05, max:0.95, step:0.05}

print(evaluer(rf, X_test, y_test, seuil=SEUIL, nom=f"Test (seuil={SEUIL})"))
y_pred_seuil = (rf.predict_proba(X_test)[:, 1] >= SEUIL).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_seuil, display_labels=["Inactif", "Toxique"],
                                        cmap="Oranges")
plt.title(f"Matrice de confusion (seuil = {SEUIL})")
plt.show()

In [ ]:
#@title 💡 Solution : rappel et précision pour tous les seuils
seuils = np.arange(0.05, 0.96, 0.05)
courbe = pd.DataFrame([evaluer(rf, X_test, y_test, seuil=s) for s in seuils], index=seuils.round(2))
courbe[["précision", "rappel", "F1"]].plot(marker="o", figsize=(7, 4))
plt.xlabel("Seuil de décision")
plt.title("Compromis précision / rappel")
plt.show()
print("Seuil qui maximise le F1 :", courbe["F1"].idxmax())

---
## 5. Quelles variables comptent ? (Feature importance) ▶️

Deux mesures complémentaires :
- **Importance par impureté** (Gini) : calculée pendant l'entraînement, rapide mais biaisée en faveur
  des variables continues ;
- **Importance par permutation** : on **mélange** une variable dans le jeu de test et on mesure la
  **perte d'AUC**. Plus la perte est grande, plus la variable est utile.

In [ ]:
imp_gini = pd.Series(rf.feature_importances_, index=COLONNES_X).sort_values()

perm = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=10,
                              random_state=42, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=COLONNES_X).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
imp_gini.plot(kind="barh", ax=axes[0], color="tab:green")
axes[0].set_title("Importance par impureté (Gini)")
imp_perm.plot(kind="barh", ax=axes[1], color="tab:purple",
              xerr=pd.Series(perm.importances_std, index=COLONNES_X)[imp_perm.index])
axes[1].set_title("Importance par permutation (perte d'AUC)")
plt.tight_layout()
plt.show()
print("Top 5 (permutation) :", imp_perm.sort_values(ascending=False).head(5).round(3).to_dict())

### 🔎 Question 5
Quelles sont les **3 variables les plus importantes** ? Donnez une **explication biologique** :
pourquoi la planéité / l'aromaticité / la lipophilie d'une molécule favoriseraient-elles l'activation
du récepteur AhR ?

⚠️ *Importance ≠ causalité* : une variable importante est **associée** à la toxicité dans ces données,
elle n'en est pas forcément la cause.

*Votre réponse :* …

---
## 6. Utiliser le modèle sur de nouvelles molécules ▶️ ✍️

In [ ]:
nouvelles = {
    "Dioxine (TCDD)": "C1=C2C(=CC(=C1Cl)Cl)OC3=CC(=C(C=C3O2)Cl)Cl",
    "Benzo[a]pyrène": "C1=CC=C2C3=C4C(=CC2=C1)C=CC5=C4C(=CC=C5)C=C3",
    "β-naphtoflavone": "C1=CC=C(C=C1)C2=CC(=O)C3=C(O2)C=CC4=CC=CC=C43",
    "Caféine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "Aspirine": "CC(=O)Oc1ccccc1C(=O)O",
    "Paracétamol": "CC(=O)Nc1ccc(O)cc1",
    # ✍️ Ajoutez votre molécule (cherchez son SMILES sur PubChem : https://pubchem.ncbi.nlm.nih.gov)
}

X_new = pd.DataFrame([calculer_descripteurs(s) for s in nouvelles.values()], index=nouvelles.keys())
resultats = pd.DataFrame({"P(toxique)": rf.predict_proba(X_new[COLONNES_X])[:, 1].round(2),
                          "cycles_aromatiques": X_new["cycles_aromatiques"],
                          "logP": X_new["logP"].round(2),
                          "halogenes": X_new["halogenes"]})

# La molécule est-elle déjà dans Tox21 ? (comparaison des SMILES écrits sous forme canonique)
canonique = lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
etiquettes_tox21 = dict(zip(data["smiles"].apply(canonique), data["toxique"]))
resultats["étiquette_Tox21"] = [etiquettes_tox21.get(canonique(s), "absente") for s in nouvelles.values()]
display(resultats.sort_values("P(toxique)", ascending=False))
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in nouvelles.values()], molsPerRow=6,
                     subImgSize=(180, 150), legends=list(nouvelles.keys()))

### 🔎 Question 6
1. Comparez `P(toxique)` et la colonne `étiquette_Tox21` (résultat expérimental, s'il existe).
   Que penser du score du **paracétamol** ?
2. Ces molécules « nouvelles » étaient en fait **déjà dans Tox21**, donc peut-être dans le jeu
   d'apprentissage ! Pourquoi faut-il toujours vérifier qu'une molécule à prédire n'a pas été vue
   pendant l'entraînement ? Ajoutez une molécule **absente** de Tox21 (cherchez son SMILES sur PubChem).

*Votre réponse :* …

---
## 7. 🏁 Livrable : un pipeline réutilisable

On regroupe toutes les étapes dans **une seule fonction** : on peut alors relancer l'analyse sur
n'importe quel test de Tox21 en une ligne.

In [ ]:
# ▶️ Pipeline complet : nettoyage → descripteurs → modèle → évaluation → importance
def pipeline_toxicite(df_source, cible, n_arbres=200, profondeur_max=12, afficher=True):
    d = df_source[["smiles", cible]].dropna().rename(columns={cible: "toxique"})
    d["toxique"] = d["toxique"].astype(int)
    d["smiles"] = d["smiles"].apply(plus_gros_fragment)
    d = d.drop_duplicates(subset="smiles")
    desc = d["smiles"].apply(calculer_descripteurs)
    d = d[desc.notna()]
    Xd = pd.DataFrame(desc.dropna().tolist(), index=d.index)
    Xtr, Xte, ytr, yte = train_test_split(Xd, d["toxique"], test_size=0.2,
                                          stratify=d["toxique"], random_state=42)
    modele = RandomForestClassifier(n_estimators=n_arbres, max_depth=profondeur_max,
                                    class_weight="balanced", random_state=42, n_jobs=-1)
    modele.fit(Xtr, ytr)
    scores = evaluer(modele, Xte, yte, nom=cible)
    importance = pd.Series(modele.feature_importances_, index=Xd.columns).sort_values(ascending=False)
    if afficher:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        ConfusionMatrixDisplay.from_estimator(modele, Xte, yte, cmap="Blues", ax=axes[0],
                                              display_labels=["Inactif", "Toxique"])
        axes[0].set_title(f"{cible} — AUC = {scores['AUC_ROC']:.2f}")
        importance.head(6).sort_values().plot(kind="barh", ax=axes[1], color="tab:green")
        axes[1].set_title("Top 6 des variables")
        plt.tight_layout()
        plt.savefig(f"rapport_{cible}.png", dpi=120)
        plt.show()
    return modele, scores, importance

_, scores_ahr, _ = pipeline_toxicite(df_brut, "NR-AhR")
print(scores_ahr)

### ✍️ Défi 1 — Comparer plusieurs tests de toxicité
Utilisez `pipeline_toxicite` pour au moins **3 autres cibles** (`"SR-MMP"`, `"SR-p53"`, `"NR-ER"`…)
et rassemblez les scores dans un tableau. Quelle toxicité est la plus facile à prédire avec ces
descripteurs simples ? Pourquoi selon vous ?

In [ ]:
# ✍️ À vous !
cibles_a_comparer = ["NR-AhR"]   # ← ajoutez d'autres cibles dans la liste

tableau = pd.concat([pipeline_toxicite(df_brut, c, afficher=False)[1] for c in cibles_a_comparer], axis=1)
tableau.T.sort_values("AUC_ROC", ascending=False)

In [ ]:
#@title 💡 Solution du défi 1
cibles_a_comparer = ["NR-AhR", "SR-MMP", "SR-p53", "NR-ER", "SR-ARE", "NR-AR"]
tableau = pd.concat([pipeline_toxicite(df_brut, c, afficher=False)[1] for c in cibles_a_comparer], axis=1)
tableau.T.sort_values("AUC_ROC", ascending=False)

### ✍️ Défi 2 — Sur-apprentissage
Remontez à la cellule 4.3, mettez `PROFONDEUR_MAX = 30` puis `PROFONDEUR_MAX = 3`, ré-exécutez les
cellules 4.3 et l'évaluation. Comparez l'AUC **apprentissage** vs **test** dans les deux cas.
Lequel **sur-apprend** ? Lequel **sous-apprend** ?

In [ ]:
#@title 💡 Solution du défi 2 : courbe de complexité
profondeurs = [2, 3, 4, 6, 8, 10, 12, 15, 20, 30]
lignes = []
for p in profondeurs:
    m = RandomForestClassifier(n_estimators=100, max_depth=p, class_weight="balanced",
                               random_state=42, n_jobs=-1).fit(X_train, y_train)
    lignes.append({"profondeur": p,
                   "AUC apprentissage": roc_auc_score(y_train, m.predict_proba(X_train)[:, 1]),
                   "AUC test": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})
pd.DataFrame(lignes).set_index("profondeur").plot(marker="o", figsize=(7, 4))
plt.ylabel("AUC ROC")
plt.title("Plus l'arbre est profond, plus l'écart apprentissage/test se creuse")
plt.show()

---
## 📝 Bilan de la séance

| Ce que vous avez fait | Notion clé |
|---|---|
| Nettoyé un CSV réel (non testés, sels, doublons) | qualité des données |
| Transformé une structure chimique en nombres | **descripteurs / features** |
| Entraîné un Random Forest | **apprentissage supervisé**, ensemble d'arbres |
| Évalué sur un jeu de test séparé | **généralisation**, sur-apprentissage |
| Lu une matrice de confusion, ajusté le seuil | **rappel vs précision**, classes déséquilibrées |
| Identifié les variables déterminantes | **interprétabilité**, importance ≠ causalité |

**Livrable à rendre :** ce notebook exécuté, avec vos réponses aux questions 🔎 1 à 6, et la figure
`rapport_NR-AhR.png`.

➡️ **Séance suivante :** que faire quand on n'a **pas d'étiquettes** ? Apprentissage non supervisé sur
des données d'expression génique.